# മനുഷ്യൻ-ഇൻ-ദി-ലൂപ്പ്: പ്രീ-ആക്ഷൻ ഗേറ്റുകൾ, റിസ്‌ക് ടിയറിംഗ്, ആഡ്‌ഡിറ്റ് ലോഗിംഗ്

ഈ പാഠത്തിന്റെ README മനുഷ്യൻ-ഇൻ-ദി-ലൂപിനെ പരിചയപ്പെടുത്തി, ഒരു ചെറിയ സ്നിപ്പെറ്റിൽ ഏജന്റ് ഇതിനകം പ്രതികരണം നൽകിയതിനു ശേഷം ഉപഭോക്താവിനെ `APPROVE` അല്ലെങ്കിൽ `REJECT` ചെയ്യാൻ ചോദിക്കുന്നു. ആ മാതൃക ഒരു നല്ല ആരംഭം ആയിരിക്കും, പക്ഷെ പ്രൊഡക്ഷൻ HITL നടപ്പാക്കലുകളിൽ സാധാരണയായി മൂന്ന് അധിക ഘടകങ്ങൾ വേണം:

1. ഏജന്റ് അപകടകരമായ ഒരു പടി നടത്തുന്നതിന് **മുന്‍പ്** പ്രവർത്തിക്കുന്ന ഒരു **പ്രീ-ആക്ഷൻ ഗേറ്റ്**, അതിലൂടെ ചെലവ്, മാറ്റാനാകാത്തതും, വൈകിപ്പോകലും നിയന്ത്രണത്തില്‍ സൂക്ഷിക്കാം.
2. **റിസ്‌ക് ടിയറിംഗ്**, അതിലൂടെ കുറഞ്ഞ റിസ്‌ക് ഉള്ള പ്രവർത്തനങ്ങൾ സ്വയമേവ നടക്കുകയും, മധ്യ റിസ്‌ക് ഉള്ള പ്രവർത്തനങ്ങൾ ബാച്ച് അംഗീകാരം ലഭിക്കുകയും, ഉയർന്ന റിസ്‌ക് ഉള്ള പ്രവർത്തനങ്ങൾക്കു മാത്രമേ മനുഷ്യന്റെ തടസ്സം വരൂ.
3. ഒരു **ഓഡിറ്റ് ലോഗും പരിഷ്കരണ ലൂപ്പും**, അതിലൂടെ ഓരോ ഗേറ്റ് തീരുമാനവും JSONL ആയി രേഖപ്പെടുത്തും, ഒരു നിരസനം ഏജന്റിനെ `Revising...` മാത്രം പ്രിന്റ് ചെയ്യുന്നതിന് പകരം സംരചനാപരമായ കാരണം ഉണ്ട് എന്ന നിലയിൽ വീണ്ടും പ്രേരിപ്പിക്കും.

ഈ നോട്ട്‌ബുക്ക് `06-system-message-framework.ipynb` ൽ ഉള്ള സമാന പ്രിമിറ്റീവുകളിൽ നിന്ന് ഓരോതും നിർമ്മിക്കുന്നു. ഇത് `DEMO_MODE = True` (ഇന്ററാക്ടീവ് ഇൻപുട്ട് ആവശ്യമില്ല) എന്ന നിലയിൽ അല്ലെങ്കിൽ യഥാർത്ഥ `input()` പ്രോപ്റ്റുകളോടെ `DEMO_MODE = False` ആയപ്പോൾ എന്റു-ടു-എൻഡ് പരിപാലിക്കുന്നു. കുറിപ്പ്: DEMO_MODE ൽ മൂന്നാം ലക്ഷ്യത്തിന്റെ റിട്രൈ സ്ക്രിപ്റ്റുചെയ്‌തതുകൊണ്ടു ലൂപ്പ് സാങ്കേതികവിദ്യകൾ മുഴുവൻ ദൃശ്യമാകും. യഥാർത്ഥ പരിഷ്കരണ ആശ്രിത പുനഃവർഗ്ഗീകരണം `DEMO_MODE = False` ആണെങ്കിൽ മാത്രം സാധിക്കും, ആ ഓപ്പറേറ്ററുകളുണ്ടായിരിക്കണം.

**പരിധിക്കടന്നത് (മറ്റു പാഠങ്ങളിൽ കൈകാര്യം ചെയ്യുന്നവ):** ഓതന്റിക്കേഷൻ, ആക്‌സസ് കൺട്രോൾ (പാഠം 06 README ഭീഷണി 2), ടൂൾ-കോൾ മിഡിൽവെയർ (പാഠം 14 MAF ഡീപ്പ് ഡൈവ്), മൾട്ടി-ഏജന്റ് ഡിബേറ്റ് പാറ്റേണുകൾ.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## പാറ്റേൺ 1: പ്രീ-ആക്‌ഷൻ ഗേറ്റ്

README-യിലെ HITL സ്നിപ്പെറ്റ് ഏജന്റ് ആദ്യം വിളിക്കുന്നു, തുടർന്ന് ഔട്ട്‌പുട്ട് അംഗീകരിക്കാൻ ഉപയോക്താവിനെ ചോദിക്കുന്നു. അത് ഒരു **പോസ്റ്റ്-ആക്‌ഷൻ** ഫ്ലോയാണ്. ഏജന്റ് ഇതിനകം പ്രവർത്തിച്ചിട്ടുണ്ട്, അതിനാൽ LLM വിളിക്കലിന്റെ ചെലവ് ഇതിനകം അടച്ചിട്ടുണ്ട്, മെയിൽ അയച്ചുകഴിഞ്ഞു, ഡാറ്റാബേസ് വരി എഴുതിയാണ്, കമന്റ് പോസ്റ്റ് ചെയ്തത് എന്നിവ സംഭവിച്ചിട്ടുണ്ട്.

ഒരു **പ്രീ-ആക്‌ഷൻ** ഫ്ലോ എജന്റ് അപകടകാരിയായ ഘട്ടത്തിലേക്ക് പോകുന്നതിന് മുമ്പ് ഗേറ്റ് സെറ്റ് ചെയ്യുന്നു. ഏജന്റ് നടപടി നിർദ്ദേശിക്കുന്നു, ഗേറ്റ് അത് നടപ്പിലാക്കണോ എന്നു നിർണ്ണയിക്കുന്നു, അംഗീകാരം ലഭിക്കുന്നത് മാത്രം സൈഡ് ഇഫക്റ്റ് നടക്കുന്നു.

| വശം | പോസ്റ്റ്-ആക്‌ഷൻ അംഗീകാരം (README സ്നിപ്പെറ്റ്) | പ്രീ-ആക്‌ഷൻ ഗേറ്റ് (ഈ നോട്ട്ബുക്ക്) |
|---|---|---|
| അംഗീകാരം എപ്പോൾ നടത്തപ്പെടുന്നു? | ഏജന്റ് ഔട്ട്‌പുട്ട് നൽകിയ ശേഷം | ഏതെങ്കിലും സൈഡ്-ഇഫക്ട് നടക്കുന്നതിന് മുമ്പ് |
| നിരസிப்பിൽ LLM ചെലവ് | ഇതിനകം അടച്ചിട്ടുണ്ട് | നിർദ്ദേശത്തിന് മാത്രം അടക്കുന്നchado, not the action |
| തിരുത്താനാകാത്ത സൈഡ് ഇഫക്ടുകൾ | സാധ്യതയുണ്ട് (നടപടി ഇതിനകം സംഭവിച്ചിട്ടുണ്ട്) | തടയപ്പെട്ടിട്ടുണ്ട് |
| ഓഡിറ്റ് വ്യക്തത | അംഗീകാരം പ്രിന്റ് സ്റ്റേറ്റ്‌മെന്റാണ് | അംഗീകാരം ടൈംസ്റാമ്പ്, നടപടി, കാരണം എന്നിവയുള്ള JSON റെക്കോർഡ് ആണ് |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## മാതൃക 2: റിസ്ക് നിരക്കുറിപ്പ്

എല്ലാ പ്രവർത്തനത്തിനും മനുഷ്യ അംഗീകാരം ആവശ്യമില്ല. ഒരു പൊതു API-യിലുള്ള വായ_only ലുക്കപ്പ്, ഒരു ഉപഭോക്തൃ ഇമെയിൽ അയക്കുന്നതിന് വ്യത്യസ്തമാണ്. ഇരട്ടത്തെയും സമാനമായി കൈകാര്യം ചെയ്യുന്നത് ഓപ്പറേറ്റേഴ്സിന്റെ ശ്രദ്ധ നശിപ്പിക്കുകയും ഏജന്റിന്റെ പ്രവർത്തനം മന്ദഗതിയാക്കുകയും ചെയ്യുന്നു.

ഒരു ലളിതമായ 3-നില മോഡൽ:

| നിര | ഉദാഹരണങ്ങൾ | അംഗീകാരം പ്രവാഹം |
|---|---|---|
| `താഴ്ന്നത്` (വായ_only) | ഒരു അറിവ് അടിസ്ഥാനത്തിൽ തിരയൽ, വിമാന ഓപ്ഷനുകൾ നോക്കൽ, ഒരു പൊതു വെബ് പേജ് ലഭ്യമാക്കൽ | ഓട്ടോ-നിർവഹണം, ഓഡിറ്റിനായി ലോഗ് ചെയ്‌തു |
| `മധ്യസ്ഥത` (ചിലവ് കുറഞ്ഞ മാറ്റം) | ഫലം ക്യാഷിങ്, കൌണ്ടർ വർദ്ധിപ്പിക്കൽ, ഓർമ്മിപ്പിക്കൽ നിശ്ചയിക്കൽ | ഓട്ടോ-നിർവഹണം, എന്നാൽ ദിവസേന ബാച്ച് അവലോകനം |
| `ഉയർന്നത്` (ബാഹ്യ-നോക്കൻശേഷിയുള്ള അല്ലെങ്കിൽ തിരികെ പോകരുതാത്ത) | ഇമെയിൽ അയയ്‌ക്കൽ, നിരക്ക് ചാർജ് ചെയ്യൽ, പൊതു ചാനലിൽ പോസ്റ്റ് ചെയ്യൽ | മനുഷ്യ അംഗീകാരത്തിന് തടയൽ |

ഇതാണ് ഒരു നിരക്കുറിപ്പ്. ഉത്പാദന സംവിധാനങ്ങളിൽ സാധാരണയായി കൂടുതൽ വിശദമായ നിരക്കുകൾ ഉപയോഗിക്കും (ഉദാഹരണം: AWS IAM അനുവാദനിരക്കുകൾ, റോളിന്റെ അടിസ്ഥാനത്തിലെ ആക്‌സസ് നിരക്കുകൾ). താഴെ കാണുന്ന 3-നില പതിപ്പ് വായ_only പ്രവർത്തനങ്ങളും സൈഡ്-ഇഫക്റ്റിംഗ് പ്രവർത്തനങ്ങളും മിശ്രിതമാക്കിയ ഏജന്റിന് ഏറ്റവും ചെറുതും പ്രയോജനകരവുമായ പതിപ്പാണ്.

താഴെ കാണുന്ന ക്ലാസിഫയർ കീവേഡ് ഹ്യൂറിസ്റ്റിക്സ് ഉപയോഗിക്കുന്നു, അതിനാൽ ഡെമോ നിർണ്ണായകവും ചെലവുകുറവുമായിരിക്കണം. ഉത്പാദന സംവിധാനത്തിൽ ഇത് പകരം പഠിച്ച ക്ലാസിഫയർ അല്ലെങ്കിൽ പോളിസി എൻജിൻ ഉപയോഗിക്കും.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## പാറ്റേൺ 3: ഓഡിറ്റ് ലോഗും പുനഃപരിശോധന ലൂപ്പും

`print("Response approved.")` ഒരു ഓഡിറ്റ് ലോഗ് അല്ല. വിശ്വാസത്തിനായി, ഓരോ ഗേറ്റ് തീരുമാനം കൂടിയും നിങ്ങൾക്ക് പിന്നീട് ചോദിക്കാനും, പുനരാവർത്തനവും, ഇൻസിഡന്റ് റിവ്യൂവിനുമായി ലഗ്ഗിംഗ് ചെയ്യാനും കഴിയുന്ന ഘടനാപരമായ ഇവന്റ് ആയി രേഖപ്പെടുത്തണം.

രണ്ടു ഭാഗങ്ങൾ:

1. **അപ്‌പെൻഡ്-ഒൺലി JSONL.** ഓരോ തീരുമാനം ഒരു ലൈൻ, ടൈംസ്റ്റാമ്പ്, ആക്ഷനും, ടിയറും, തീരുമാനവും, കാരണം. grep ചെയ്യാനും, പിന്നീട് യഥാർത്ഥ ലോഗ് സ്റ്റോറിലേയ്ക്ക് അയക്കാനും എളുപ്പം.
2. **പ്രതിഷേധത്തിൽ പുനഃപരിശോധന ലൂപ്പ്.** ഗേറ്റ് `deny` മൊഴിയുമ്പോൾ, ഏജന്റ് നിരസിച്ച കാരണം കോൺടെക്സ്റ്റിൽ നൽകിയുകൊണ്ട് സ്വയം വീണ്ടും പ്രോംപ്റ്റ് ചെയ്യും, അതിനാൽ അടുത്ത നിർദ്ദേശം പ്രശ്‌നം ഒഴിവാക്കാൻ കഴിയും.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## അധിക വിഭവങ്ങൾ

ഈ HITL نمുനകളുടെ വിവിധ വേരിയേഷൻസുകൾ പ്രവർത്തിപ്പിക്കുന്ന മറ്റു പല പൊതുജന പ്രോജക്റ്റുകളും ഉണ്ട്. നിങ്ങളുടെ സ്റ്റാക്കിനൊപ്പം ഏത് അനുയോജ്യമാണ് എന്ന് കണ്ടെത്താൻ സമീപനങ്ങൾ താരതമ്യം ചെയ്യുക:

- **LangChain** human-in-the-loop ടൂൾ റാപ്പറുകൾ ([ഡോക്സ്](https://python.langchain.com/docs/integrations/tools/human_tools)): മനുഷ്യന്റെ ഇൻപുട്ടിനായി എക്സിക്യൂഷൻ നിലക്കുന്നതിനുള്ള ഡ്രോപ്പ് ഇൻ ടൂൾ റാപ്പറുകൾ.
- **AutoGen** `UserProxyAgent` ([v0.2 ഡോക്സ്](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ഇത് പുനസംഘടിപ്പിച്ചു): ബഹുഏജന്റ് സംഭാഷണങ്ങളിൽ മനുഷ്യനെ പ്രതിനിധാനം ചെയ്യുന്നതിന് പ്രത്യേകിച്ചുള്ള ഒരു ഏജന്റ് റോളിനെ ഉപയോഗിക്കുന്നു.
- **Semantic Kernel** ഫംഗ്ഷൻ ഫിൽട്ടറുകൾ ([ഡോക്സ്](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): ഓരോ ഫംഗ്ഷൻ കോൾ ചുറ്റും പ്രവർത്തിക്കുന്ന മിഡിൽവെയർ, ഗേറ്റിംഗ് ലോജിക്കിന് അനുയോജ്യം.

ഓരോ പ്രോജക്റ്റും ഈ മൂന്ന് ഉപ-നമുക്കളെയും വ്യത്യസ്തമായി കൈകാര്യം ചെയ്യുന്നു: LangChain അവയെ ടൂളുകളായി റാപ്പ് ചെയ്യുന്നു, AutoGen ഏജന്റ് റോളിനെ ഉപയോഗിക്കുന്നു, Semantic Kernel മിഡിൽവെയർ ഫിൽട്ടറുകൾ ഉപയോഗിക്കുന്നു. നിങ്ങളുടെ സ്വന്തം ഏജന്റിന് ഡിസൈൻ തിരഞ്ഞെടുക്കുന്നതിന് മുമ്പായി ഒന്നോ രണ്ടോ ഇംപ്ലിമെന്റേഷനുകൾ ആരംഭത്തിൽ നിന്നു അവസാനം വരെ വായിക്കുക.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
